# This notebook is to play with my agent

In [1]:
from typing import Optional
from google.adk.agents import LlmAgent
from google.adk.tools import ToolContext
from google.adk.tools.preload_memory_tool import PreloadMemoryTool
from google.adk.runners import InMemoryRunner
import asyncio
from dotenv import load_dotenv
from google.genai.types import Content, Part
import json

load_dotenv()

# Step 1: Create the agent.
mindy_agent = LlmAgent(
    name="nutritionist",
    model="gemini-2.5-flash",
    instruction=(
        """
        You are Mindy, a compassionate nutritionist specializing in the MIND diet. 
        Your mission is to help users achieve their health goals by recommending 
        meal plans and complete recipes that follow MIND diet principles. 
        You actively listen to users, ask about their milestones, feelings, 
        and personal goals, and use this information to provide personalized, 
        motivating, and supportive guidance. Always encourage users, celebrate their 
        progress, and adapt your recommendations to their unique needs 
        and preferences.
        """
    ), 
    tools = [PreloadMemoryTool()],
)

# Create parameters
runner = InMemoryRunner(agent=mindy_agent)
selected_foods = ["oatmeal", "beans"]
user_id = "001"
session_id = "session1"
foods_str = ", ".join(selected_foods)
prompt = (
        f"Based on the following selected foods: {foods_str}, "
        "please create a weekly meal plan following the mind diet."
        "Include complete recipes for each meal."
        "Return the result as a JSON object in the following format:\n"
        "{\n"
        '  "week": [\n'
        '    {\n'
        '      "day": "Monday",\n'
        '      "meals": [\n'
        '        {\n'
        '          "name": "Breakfast",\n'
        '          "recipe": {\n'
        '            "title": "Oatmeal with Berries",\n'
        '            "ingredients": ["oats", "blueberries", "almond milk"],\n'
        '            "instructions": "Mix oats with almond milk, cook, and top with berries."\n'
        '          }\n'
        '        },\n'
        '        ...\n'
        '      ]\n'
        '    },\n'
        '    ...\n'
        '  ]\n'
        "}\n"
        "Only return valid JSON."
)

message = Content(
        role="user",
        parts=[Part(text=prompt)]
)

async_create_session = await runner.session_service.create_session(
        app_name=runner.app_name,
        user_id=user_id,
        session_id=session_id
    )


# async for event in runner.run_async(
#     user_id = user_id,
#     session_id = async_create_session.id,
#     new_message = message
# ): 


In [2]:
events_list = runner.run_async(
     user_id = user_id,
     session_id = async_create_session.id,
     new_message = message
)

In [13]:
event.content

Content(
  parts=[
    Part(
      text="""```json
{
  "week": [
    {
      "day": "Monday",
      "meals": [
        {
          "name": "Breakfast",
          "recipe": {
            "title": "Berry & Walnut Oatmeal",
            "ingredients": [
              "1/2 cup rolled oats",
              "1 cup water or unsweetened almond milk",
              "1/2 cup mixed berries (fresh or frozen)",
              "1/4 cup walnuts, chopped",
              "1 tsp chia seeds (optional)",
              "A pinch of cinnamon"
            ],
            "instructions": "Combine oats, water/almond milk, and cinnamon in a small saucepan. Bring to a boil, then reduce heat and simmer for 5-7 minutes, stirring occasionally, until liquid is absorbed and oats are creamy. Transfer to a bowl and top with mixed berries, chopped walnuts, and chia seeds."
          }
        },
        {
          "name": "Lunch",
          "recipe": {
            "title": "Mediterranean Chickpea Salad",
            "ingred

In [5]:
async for event in events_list:
    print(event)

model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""```json
{
  "week": [
    {
      "day": "Monday",
      "meals": [
        {
          "name": "Breakfast",
          "recipe": {
            "title": "Berry & Walnut Oatmeal",
            "ingredients": [
              "1/2 cup rolled oats",
              "1 cup water or unsweetened almond milk",
              "1/2 cup mixed berries (fresh or frozen)",
              "1/4 cup walnuts, chopped",
              "1 tsp chia seeds (optional)",
              "A pinch of cinnamon"
            ],
            "instructions": "Combine oats, water/almond milk, and cinnamon in a small saucepan. Bring to a boil, then reduce heat and simmer for 5-7 minutes, stirring occasionally, until liquid is absorbed and oats are creamy. Transfer to a bowl and top with mixed berries, chopped walnuts, and chia seeds."
          }
        },
        {
          "name": "Lunch",
          "recipe": {
            "title": "Mediterra

In [14]:
type(event)

google.adk.events.event.Event

In [15]:
type(event.content)

google.genai.types.Content

In [5]:
event.content.parts[0].text

'```json\n{\n  "week": [\n    {\n      "day": "Monday",\n      "meals": [\n        {\n          "name": "Breakfast",\n          "recipe": {\n            "title": "MINDful Berry Oatmeal",\n            "ingredients": [\n              "1/2 cup rolled oats",\n              "1 cup unsweetened almond milk (or water)",\n              "1/2 cup mixed berries (fresh or frozen)",\n              "1 tablespoon chopped walnuts",\n              "1/2 teaspoon cinnamon"\n            ],\n            "instructions": "Combine oats and almond milk (or water) in a small saucepan. Bring to a simmer over medium heat and cook for 5-7 minutes, stirring occasionally, until oats are creamy. Pour into a bowl, top with mixed berries, walnuts, and a sprinkle of cinnamon. Enjoy warm."\n          }\n        },\n        {\n          "name": "Lunch",\n          "recipe": {\n            "title": "Mediterranean Chickpea Salad",\n            "ingredients": [\n              "2 cups mixed greens",\n              "1/2 can (7.

In [3]:
response = event.content.parts[0].text.strip('```json').strip('```').strip()


In [4]:
parsed = json.loads(response)

In [28]:
parsed

{'week': [{'day': 'Monday',
   'meals': [{'name': 'Breakfast',
     'recipe': {'title': 'Blueberry Almond Oatmeal',
      'ingredients': ['1/2 cup rolled oats',
       '1 cup water or unsweetened almond milk',
       '1/2 cup fresh blueberries',
       '1/4 cup sliced almonds',
       'Pinch of cinnamon'],
      'instructions': 'Combine rolled oats and water/almond milk in a saucepan. Bring to a boil, then reduce heat and simmer for 5-7 minutes, stirring occasionally, until desired consistency is reached. Stir in cinnamon. Top with fresh blueberries and sliced almonds.'}},
    {'name': 'Lunch',
     'recipe': {'title': 'Mediterranean Chickpea Salad',
      'ingredients': ['1 (15-ounce) can chickpeas, rinsed and drained',
       '1/2 cucumber, diced',
       '1 cup cherry tomatoes, halved',
       '1/4 red onion, thinly sliced',
       '1/2 bell pepper (any color), diced',
       '1/4 cup fresh parsley, chopped',
       '1/4 cup Kalamata olives, halved',
       '2 tablespoons extra virg

In [12]:
response = json.loads(str(event))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)